# 🎯 Computer Vision: Image Segmentation with YOLO & SAM 2
### Complete Hands-On Teaching Notebook (Fully Commented for Students)

---

### 🧠 What is Image Segmentation?
- **1. Image Classification:** Answers *"What is in this picture?"* (e.g., Output: `"person"`)
- **2. Object Detection:** Answers *"Where is the object?"* (e.g., Output: Bounding Box `[x_min, y_min, x_max, y_max]`)
- **3. Image Segmentation:** Answers *"Which exact pixels belong to the object?"* (e.g., Output: Pixel-level binary mask where `1` = object, `0` = background)

---

### 🛠️ The 2 State-of-the-Art Models We Teach Today:
| Model | Creator | Type | Key Advantage |
| :--- | :--- | :--- | :--- |
| **🚀 YOLO (YOLO11-seg)** | Ultralytics | Fully Automatic | **Ultra fast (100+ FPS)**, recognizes 80 everyday classes |
| **🔮 SAM 2 (Segment Anything 2)** | Meta AI | Prompt-Based | **Zero-shot (segments any object in the world)** via point clicks or boxes |

---
## 📦 Step 1: Install Libraries & Download Sample Image
We use `ultralytics` which provides **both YOLO11 and SAM 2** in one unified Python package.

In [ ]:
# Install required packages:
# - ultralytics: Official library for YOLO models and built-in SAM 2
# - opencv-python: Image manipulation, color conversion, blurring
# - matplotlib: Plotting and displaying images inside the notebook
# Note: '-q' means quiet mode (hides lengthy terminal download logs)
!pip install -q ultralytics opencv-python matplotlib

# Download a sample teaching image containing people and a bus:
# - '-q': quiet download
# - '-O bus.jpg': save the downloaded file locally as 'bus.jpg'
!wget -q -O bus.jpg https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg

print("✅ Setup complete! Libraries installed and 'bus.jpg' downloaded.")

---
## 🚀 Part 1: YOLO Segmentation (Automatic & Real-Time)
YOLO detects objects, recognizes their category, and calculates pixel masks in **a single forward pass**.

In [ ]:
from ultralytics import YOLO             # Import YOLO class from Ultralytics
import matplotlib.pyplot as plt          # Import Matplotlib for image display

# Parameter Explanation for YOLO():
# - 'yolo11n-seg.pt':
#     'yolo11' = YOLO version 11 (latest architecture)
#     'n'      = Nano model (smallest size ~6MB, fastest speed, low memory)
#     '-seg'   = Trained for Instance Segmentation (not just detection)
#     '.pt'    = PyTorch model weights file (downloaded automatically)
model = YOLO("yolo11n-seg.pt")

# Parameter Explanation for model():
# - source='bus.jpg': Path to the input image (can also be a URL, numpy array, or video)
# - conf=0.35: Confidence threshold (ignores detections below 35% probability)
# - verbose=False: Suppresses verbose frame-by-frame console logging
results = model("bus.jpg", conf=0.35, verbose=False)

# Visualizing the output:
# - results[0]: Results for the first image in our batch
# - .plot(): Renders the original image with colored segmentation masks, bounding boxes, and labels
annotated_image = results[0].plot()

plt.figure(figsize=(10, 6))               # Set figure display size (width=10, height=6 inches)
plt.imshow(annotated_image)              # Display the annotated image array
plt.axis("off")                          # Hide X and Y axis coordinate ticks for clean visualization
plt.title("YOLO11: Automatic Real-Time Segmentation", fontsize=14, fontweight='bold')
plt.show()                               # Render the plot

### 🔍 Inspecting the 3 Core Outputs of Segmentation
Under the hood, `results[0]` gives us 3 key PyTorch tensors:
1. **Bounding Boxes:** `res.boxes.xyxy`
2. **Class IDs & Names:** `res.boxes.cls` and `model.names`
3. **Binary Segmentation Masks:** `res.masks.data`

In [ ]:
res = results[0]  # Grab the primary result object

# 1. Bounding Boxes Tensor:
# Shape: [Number_of_Objects, 4]
# Format: [x_min, y_min, x_max, y_max] in pixel coordinates
print("1. Bounding Boxes (x_min, y_min, x_max, y_max):\n", res.boxes.xyxy)

# 2. Detected Class Numeric IDs:
# Shape: [Number_of_Objects] (e.g., 0=person, 5=bus in the 80 COCO classes)
print("\n2. Detected Class IDs:\n", res.boxes.cls)

# 3. Human-Readable Class Names:
# model.names is a dictionary mapping integer ID -> string label
detected_names = [model.names[int(class_id)] for class_id in res.boxes.cls]
print("\n3. Detected Class Names:\n", detected_names)

# 4. Segmentation Masks Tensor:
# Shape: [Number_of_Objects, Height, Width]
# Each mask is a 2D grid containing 1s (object pixels) and 0s (background pixels)
print("\n4. Masks Tensor Shape [Num_Objects, Height, Width]:", res.masks.data.shape)

---
## 🔮 Part 2: SAM 2 (Segment Anything 2 by Meta AI)

### How is SAM 2 different from YOLO?
- **Class-Agnostic:** SAM 2 doesn't output "person" or "car". It outputs pure pixel masks.
- **Zero-Shot:** It can segment **any object in the universe** (medical X-rays, cells, cartoons, drone images) without retraining.
- **Prompt-Driven:** You tell SAM *what* to segment using:
  1. A **Point Prompt** (clicking on an object `[x, y]`)
  2. A **Box Prompt** (providing a bounding box `[x1, y1, x2, y2]`)
  3. **No Prompt** (automatic "Segment Everything")

### 💡 How to find `(x, y)` coordinates easily?
Turn on the coordinate grid so students can read `(x, y)` directly off the image axes!

In [ ]:
import cv2

# Read image with OpenCV: cv2.imread loads in BGR color order by default
# [:, :, ::-1] reverses the 3rd channel (Blue-Green-Red -> Red-Green-Blue for Matplotlib)
img_rgb = cv2.imread("bus.jpg")[:, :, ::-1]

# Display with coordinate ruler grid:
plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.axis("on")                                                # Keep axis ticks visible to read pixel coordinates
plt.grid(True, color='white', linestyle='--', alpha=0.6)      # White dashed grid lines every 100 pixels
plt.xlabel("➡️ X Coordinate (horizontal distance from left edge)", fontsize=11, fontweight='bold')
plt.ylabel("⬅️ Y Coordinate (vertical distance from top edge)", fontsize=11, fontweight='bold')
plt.title("Coordinate Ruler: Use this to pick (X, Y) for SAM Prompts!", fontsize=13, fontweight='bold')
plt.show()

### 🎯 SAM 2 - Example A: Point Prompt
Click on an object using a single `[x, y]` point.

In [ ]:
from ultralytics import SAM               # Import SAM class from Ultralytics

# Load SAM 2 model:
# - 'sam2.1_t.pt':
#     'sam2.1' = Meta's Segment Anything Model version 2.1
#     't'      = Tiny model (fastest inference, low VRAM usage)
#     '.pt'    = Model weights downloaded automatically
sam_model = SAM("sam2.1_t.pt")

# Parameter Explanation for Point Prompt:
# - source='bus.jpg': Input image path
# - points=[[250, 400]]: List of [x, y] coordinates (X=250, Y=400 is on the person in the middle)
# - labels=[1]: Foreground/Background flag for each point:
#     1 = Positive click (Include this object)
#     0 = Negative click (Exclude this region)
point_results = sam_model("bus.jpg", points=[[250, 400]], labels=[1], verbose=False)

plt.figure(figsize=(10, 6))
plt.imshow(point_results[0].plot())
plt.axis("off")
plt.title("SAM 2: Point Prompt [X=250, Y=400]", fontsize=13, fontweight='bold')
plt.show()

### 📦 SAM 2 - Example B: Bounding Box Prompt
Specify a bounding box `[x_min, y_min, x_max, y_max]` to isolate an object.

In [ ]:
# Define bounding box around the left-most person:
# - x_min=50: Left boundary
# - y_min=390: Top boundary
# - x_max=240: Right boundary
# - y_max=890: Bottom boundary
box_coords = [50, 390, 240, 890]

# Parameter Explanation:
# - bboxes=[box_coords]: List of bounding boxes to segment
box_results = sam_model("bus.jpg", bboxes=[box_coords], verbose=False)

plt.figure(figsize=(10, 6))
plt.imshow(box_results[0].plot())
plt.axis("off")
plt.title("SAM 2: Bounding Box Prompt [50, 390, 240, 890]", fontsize=13, fontweight='bold')
plt.show()

### 🌐 SAM 2 - Example C: Segment Everything (No Prompts)
Pass no prompts — SAM 2 automatically segments **every object, part, and region** in the image.

In [ ]:
# Calling SAM without 'points' or 'bboxes' prompts triggers Automatic Mask Generation:
everything_results = sam_model("bus.jpg", verbose=False)

plt.figure(figsize=(10, 6))
plt.imshow(everything_results[0].plot())
plt.axis("off")
plt.title("SAM 2: Automatic 'Segment Everything' (All parts & objects)", fontsize=13, fontweight='bold')
plt.show()

---
## 💡 Part 3: Cheat Sheet — YOLO vs SAM 2 Comparison

| Feature | 🚀 YOLO Segmentation | 🔮 SAM 2 (Segment Anything) |
| :--- | :--- | :--- |
| **Input Needed** | None (Fully automatic) | Prompt (Point click, Bounding box, or None) |
| **Speed** | ⚡ **Ultra Fast (100+ FPS)** | 🐢 **Moderate (15 - 30 FPS)** |
| **Class Recognition** | ✅ **Yes** (Identifies "person", "bus", "car") | ❌ **No** (Class-agnostic mask only) |
| **Vocabulary** | Fixed categories (80 COCO classes) | 🌍 **Zero-Shot: Any object in the universe** |
| **Best Use-Case** | Real-time video, autonomous driving, robotics | Data labeling, medical imaging, photo editing |

### 🎯 Golden Rule for Students:
- Need **speed** + **class names**? 👉 **YOLO**
- Need **surgical control** + **unseen custom objects**? 👉 **SAM 2**

---
## 🌟 Part 4: Practical Real-World Application — Smart Portrait Mode & Virtual Background

### ❓ Why are Bounding Boxes NOT enough for this?
If you use an **Object Detection Bounding Box**, you get a rectangular crop around the person. If you try to blur the background or replace it, you end up blurring a box around them or cutting off shoulders and arms!

With **Segmentation**, we have a **pixel-perfect mask**. This enables two of the most popular computer vision features used in modern apps:
1. **📸 Portrait Mode (Bokeh Effect):** Keeps the person razor-sharp while artistically blurring the background (like an expensive DSLR lens or iPhone Portrait Mode).
2. **💻 Virtual Background (Zoom / Google Meet):** Replaces messy room backgrounds with a sleek solid studio color or virtual background in just 4 lines of NumPy!

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# --------------------------------------------------------------------
# Step 1: Load image and get pixel segmentation mask of the person
# --------------------------------------------------------------------
original_bgr = cv2.imread("bus.jpg")
original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
h, w, _ = original_rgb.shape                     # Original image height and width in pixels

# Run YOLO segmentation to find the person
seg_results = model("bus.jpg", conf=0.4, verbose=False)[0]

# Grab the mask tensor of the first detected person (Index 0)
# - .data[0]: Tensor of shape [mask_h, mask_w] with values between 0.0 and 1.0
# - .cpu().numpy(): Moves tensor from GPU to CPU as a NumPy array
raw_person_mask = seg_results.masks.data[0].cpu().numpy()

# Resize mask back to exact original image resolution (w, h):
# - cv2.INTER_NEAREST: Keeps sharp binary edges without blurring mask boundary
# - .astype(bool): Converts float values to True (person pixel) and False (background pixel)
person_mask = cv2.resize(raw_person_mask, (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)

# Expand mask shape from (Height, Width) to (Height, Width, 1) so it broadcasts with 3-channel RGB
person_mask_3d = person_mask[:, :, None]

# --------------------------------------------------------------------
# Step 2: Application 1 — Portrait Mode (Bokeh Background Blur)
# --------------------------------------------------------------------
# Blur the entire image with Gaussian Blur:
# - ksize=(55, 55): Kernel window size (must be odd numbers; larger = stronger blur)
# - sigmaX=0: Standard deviation calculated automatically from kernel size
blurred_background = cv2.GaussianBlur(original_rgb, ksize=(55, 55), sigmaX=0)

# np.where(condition, value_if_true, value_if_false):
# - Where person_mask_3d is True  -> Keep original sharp person pixels
# - Where person_mask_3d is False -> Use blurred background pixels
portrait_mode_image = np.where(person_mask_3d, original_rgb, blurred_background)

# --------------------------------------------------------------------
# Step 3: Application 2 — Virtual Background (Zoom / Teams Replacement)
# --------------------------------------------------------------------
# Create a sleek dark-blue virtual studio background (R=25, G=35, B=60)
virtual_studio_bg = np.full_like(original_rgb, fill_value=[25, 35, 60], dtype=np.uint8)

# Blend person with the virtual background:
virtual_bg_image = np.where(person_mask_3d, original_rgb, virtual_studio_bg)

# --------------------------------------------------------------------
# Step 4: Display all 3 results side-by-side for comparison
# --------------------------------------------------------------------
plt.figure(figsize=(18, 6))

# Subplot 1: Original Image
plt.subplot(1, 3, 1)
plt.imshow(original_rgb)
plt.title("1. Original Image", fontsize=13, fontweight='bold')
plt.axis("off")

# Subplot 2: Portrait Mode (Background Blur)
plt.subplot(1, 3, 2)
plt.imshow(portrait_mode_image)
plt.title("2. 📸 Portrait Mode (Background Blur)", fontsize=13, fontweight='bold', color='#0066cc')
plt.axis("off")

# Subplot 3: Virtual Background (Zoom Replacement)
plt.subplot(1, 3, 3)
plt.imshow(virtual_bg_image)
plt.title("3. 💻 Virtual Background (Zoom Style)", fontsize=13, fontweight='bold', color='#009933')
plt.axis("off")

plt.tight_layout()
plt.show()

print("✨ Real-world application executed successfully!")

---
## 🎥 Part 5: The Grand Finale — Real-Time Live Video Stream (Google Colab)

### ❓ Why doesn't standard `cv2.VideoCapture(0)` work in Google Colab?
Google Colab runs on remote servers inside Google's cloud data centers. The server has **no physical camera attached**.

### 💡 How our Real-Time Video Stream works:
1. **JavaScript in Browser:** Captures your laptop webcam feed using `navigator.mediaDevices.getUserMedia()`.
2. **Colab Kernel Bridge:** Sends video frames to Python via `google.colab.kernel.invokeFunction()`.
3. **GPU YOLO Inference:** Python runs `model(frame)` on the GPU and returns the segmented video frame.
4. **Live Stream:** Displays the real-time segmented video inside Colab at **20+ FPS** with an interactive **Stop button**!

In [ ]:
import base64                                              # For converting binary image bytes to text string
import numpy as np                                         # For array and matrix operations
import cv2                                                 # OpenCV for decoding and encoding video frames
from IPython.display import display, HTML, JSON             # For rendering custom interactive HTML/JS inside Colab

def live_video_stream(model, width=480, height=360):
    """
    Streams live webcam video from your browser into Google Colab, runs real-time
    YOLO instance segmentation on every frame, and displays the annotated video stream.
    
    Parameters:
    - model: Pre-loaded Ultralytics YOLO segmentation model (e.g. yolo11n-seg)
    - width: Video stream resolution width in pixels (default: 480)
    - height: Video stream resolution height in pixels (default: 360)
    """
    try:
        # google.colab.output is only available when running in Google Colab
        from google.colab.output import register_callback
    except ImportError:
        print("⚠️ This streaming function is designed for Google Colab!")
        print("If running locally on a laptop, use standard cv2.VideoCapture(0) below.")
        return

    # 1. Python Callback Function:
    # Called by JavaScript for every new video frame captured from the webcam
    def process_frame(data_url):
        # data_url format: 'data:image/jpeg;base64,/9j/4AAQSkZJRg...'
        # Split metadata header and extract pure base64 image string
        header, encoded_data = data_url.split(',', 1)
        image_bytes = base64.b64decode(encoded_data)        # Decode base64 text back into raw binary bytes
        np_buffer = np.frombuffer(image_bytes, np.uint8)    # Convert bytes into a 1D NumPy array
        frame = cv2.imdecode(np_buffer, cv2.IMREAD_COLOR)   # Decode bytes into a BGR image matrix

        # Run YOLO segmentation on the current webcam frame
        # - verbose=False: disables printing inference stats on every frame to keep terminal fast
        results = model(frame, verbose=False)
        annotated_frame = results[0].plot()                 # Draw segmentation masks and boxes on frame

        # Encode the annotated frame back into JPEG format:
        # - cv2.IMWRITE_JPEG_QUALITY, 65: balances high visual quality with small byte size for low latency
        _, jpeg_buffer = cv2.imencode('.jpg', annotated_frame, [cv2.IMWRITE_JPEG_QUALITY, 65])
        base64_result = base64.b64encode(jpeg_buffer).decode('utf-8')

        # Return JSON payload back to the browser JavaScript loop
        return JSON({'image': f"data:image/jpeg;base64,{base64_result}"})

    # Register callback so browser JavaScript can invoke it via kernel.invokeFunction
    register_callback('notebook.process_frame', process_frame)

    # 2. Interactive HTML5 & JavaScript Video Player with Stop Button and Live FPS Counter
    stream_ui = f"""
    <div style="font-family: Arial, sans-serif; margin-bottom: 12px;">
        <button id="stop_stream_btn" style="padding: 10px 20px; font-size: 15px; background: #dc3545; color: white; border: none; border-radius: 6px; cursor: pointer; font-weight: bold;">
            ⏹️ Stop Video Stream
        </button>
        <span id="fps_label" style="margin-left: 15px; font-size: 16px; font-weight: bold; color: #28a745;">⚡ Starting Camera...</span>
    </div>
    <div style="display: inline-block;">
        <!-- Hidden video and canvas elements used to grab frames from browser webcam -->
        <video id="webcam_feed" width="{width}" height="{height}" autoplay playsinline style="display:none;"></video>
        <canvas id="capture_canvas" width="{width}" height="{height}" style="display:none;"></canvas>
        <!-- Visible output image element showing real-time YOLO segmentation -->
        <img id="segmented_stream" width="{width}" height="{height}" style="border: 3px solid #333; border-radius: 8px; display: block;" />
    </div>

    <script>
    (async function() {{
        const video = document.getElementById('webcam_feed');
        const canvas = document.getElementById('capture_canvas');
        const ctx = canvas.getContext('2d');
        const outputImg = document.getElementById('segmented_stream');
        const stopBtn = document.getElementById('stop_stream_btn');
        const fpsLabel = document.getElementById('fps_label');

        let active = true;
        stopBtn.onclick = () => {{ active = false; }};       // Set flag to false when user clicks Stop

        try {{
            // Request webcam access from browser
            const stream = await navigator.mediaDevices.getUserMedia({{
                video: {{ width: {width}, height: {height} }}
            }});
            video.srcObject = stream;
            await video.play();
            fpsLabel.textContent = "🟢 Live YOLO Video Stream Running";

            let frames = 0;
            let prevTime = performance.now();

            // Main continuous video streaming loop
            while (active) {{
                // 1. Draw current webcam frame onto hidden canvas
                ctx.drawImage(video, 0, 0, {width}, {height});
                const frameData = canvas.toDataURL('image/jpeg', 0.6); // 60% quality for fast transfer

                // 2. Invoke Python process_frame callback asynchronously
                const response = await google.colab.kernel.invokeFunction('notebook.process_frame', [frameData], {{}});
                outputImg.src = response.data['application/json'].image;

                // 3. Calculate and display real-time FPS
                frames++;
                const now = performance.now();
                if (now - prevTime >= 1000) {{
                    const currentFps = Math.round((frames * 1000) / (now - prevTime));
                    fpsLabel.textContent = `🟢 Live Stream (FPS: ${{currentFps}})`;
                    frames = 0;
                    prevTime = now;
                }}
            }}

            // Stop camera tracks cleanly when active becomes false
            stream.getTracks().forEach(track => track.stop());
            fpsLabel.textContent = "🔴 Stream Stopped";
            fpsLabel.style.color = "#dc3545";
        }} catch (err) {{
            fpsLabel.textContent = "❌ Camera error: " + err.message;
            fpsLabel.style.color = "#dc3545";
        }}
    }})();
    </script>
    """
    display(HTML(stream_ui))

print("✅ Live video stream helper defined and ready to run!")

### 🚀 Start the Real-Time Video Stream
Run this cell in Google Colab! Your browser will ask for webcam permission, and you will see the live segmentation stream right here.

In [ ]:
# Start live video stream:
# - Pass 'model' (YOLO11-seg)
# - Click the red 'Stop Video Stream' button in the output to end the stream
live_video_stream(model)

### 💻 Optional: If Running Locally on your Laptop (VS Code / Jupyter)
If you are not running on Colab and running on a real laptop with an integrated camera, OpenCV's standard loop works directly:

In [ ]:
# (Run this only when running locally on your PC/Mac, NOT in Google Colab):
"""
cap = cv2.VideoCapture(0)                        # 0 is the default laptop webcam index
while cap.isOpened():
    success, frame = cap.read()                  # Grab the next webcam frame
    if not success: break
    
    results = model(frame, verbose=False)        # Run YOLO segmentation on the frame
    cv2.imshow("Live YOLO (Press Q to quit)", results[0].plot()) # Show in OpenCV desktop window
    
    if cv2.waitKey(1) & 0xFF == ord('q'):        # Press 'q' key to quit the loop
        break

cap.release()                                    # Release camera hardware
cv2.destroyAllWindows()                          # Close desktop window
"""
print("ℹ️ Local laptop OpenCV code available above (commented out for Google Colab)")